# Baseline Comparison — ACO vs Standard ALNS vs Hybrid ALNS

Requirement (spec §6): compare all three algorithms on both datasets under the
same computation budget, full vehicle availability, all refill stations available,
20 independent seeds per algorithm. Every run is saved (not only the best).

**Portability**: this notebook depends only on `backend/engine/*` and
`experiments/common.py` — no FastAPI, no database. See `experiments/README.md`
for what to copy if you want to run this outside the repo.

In [ ]:
# Ensure repo root is importable when the notebook is opened from experiments/.
import os, sys
_REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _REPO not in sys.path:
    sys.path.insert(0, _REPO)

from experiments.common import (
    RunSpec, run_many, write_csv, all_park_ids, FULL_FLEET,
)
from backend.engine.io_utils import load_dataset, get_available_datasets
print('Available datasets:', [s.id for s in get_available_datasets()])

## Experiment configuration

In [ ]:
DATASETS = ['dataset_a', 'dataset_b']
ALGORITHMS = ['aco', 'alns_standard', 'alns_hybrid']
SEEDS = list(range(20))       # 20 independent runs per algorithm per dataset
TIME_LIMIT_SEC = 30.0          # offline/manuscript budget (NOT the shortened demo budget)

OUT_COLUMNS = [
    'dataset', 'algorithm', 'seed', 'vehicle_count',
    'total_time', 'makespan', 'route_time_std',
    'active_vehicles', 'refill_visits',
    'computation_time', 'feasible',
]

## Build the run list

In [ ]:
available = {s.id for s in get_available_datasets()}

runs_by_dataset = {}
for dataset_id in DATASETS:
    if dataset_id not in available:
        print(f'[skip] {dataset_id} — files missing (run scripts/convert_dataset.py)')
        continue
    nodes, _tm, _spec = load_dataset(dataset_id)
    park_ids = all_park_ids(nodes)
    fleet = FULL_FLEET[dataset_id]
    specs = [
        RunSpec(
            dataset_id=dataset_id, algorithm=algo, seed=seed,
            num_vehicles=fleet, refill_ids=None,   # None = all refills available
            time_limit_sec=TIME_LIMIT_SEC,
            selected_park_ids=park_ids,
        )
        for algo in ALGORITHMS for seed in SEEDS
    ]
    runs_by_dataset[dataset_id] = specs
    print(f'{dataset_id}: {len(specs)} runs planned (fleet={fleet}, parks={len(park_ids)})')

## Run and save

Each dataset writes its own CSV (`baseline_dataset_a.csv`, `baseline_dataset_b.csv`).

Total wall time ≈ `len(specs) × TIME_LIMIT_SEC` — expect ~30 min per dataset at
the full 30-second budget × 60 runs. Cut TIME_LIMIT_SEC to 5-10s above for a quick
smoke test.

In [ ]:
RESULTS_DIR = os.path.join(_REPO, 'experiments', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

for dataset_id, specs in runs_by_dataset.items():
    print(f'\n=== {dataset_id} ({len(specs)} runs) ===')
    rows = run_many(specs, progress=True)
    out_path = os.path.join(RESULTS_DIR, f'baseline_{dataset_id}.csv')
    write_csv(rows, out_path, OUT_COLUMNS)

## Quick summary

In [ ]:
import pandas as pd

for dataset_id in runs_by_dataset.keys():
    p = os.path.join(RESULTS_DIR, f'baseline_{dataset_id}.csv')
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p)
    print(f'\n{dataset_id}:  rows={len(df)}  all-feasible={df.feasible.all()}')
    print(df.groupby('algorithm')[['makespan','total_time','route_time_std','refill_visits','computation_time']].mean().round(2))